# GeniZ: Metapath Attention Model for Technological Knowledge Pathways

**Purpose**: Identify which relational pathways structure knowledge circulation
in the European innovation system and characterise its core-periphery architecture through contextual centrality patterns (PageRank z-score).

**Pipeline**:
1. **GeniZ Lite** — exploratory model on all 14 metapaths → attention weights as signal
2. **Branch separation** — conceptual split: Rama A (knowledge) vs Rama B (institutional)
3. **Forward selection** — data-driven selection within each branch (Ridge proxy)
4. **GeniZ Final** — full model on selected metapaths
5. **Evaluation** — Spearman ρ, NDCG@100, RBP(p=0.95/0.80), R²
6. **Permutation Feature Importance** → Table 4
7. **Core-Periphery** via GMM (data-driven)

**Key design decisions**:
- Loss: HuberLoss + band weighting + BPR with inertia
- Checkpoint: best `combined = (Spearman + NDCG) / 2`
- RBP as evaluation metric only — not in training objective
- Branch separation is conceptual (metapath semantics), forward selection is data-driven

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pickle, glob, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List, Optional
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.metrics import r2_score, ndcg_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from scipy.stats import spearmanr
from tqdm import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')

## 2. Configuration

In [ ]:
BASE_DIR   = '/content/drive/MyDrive/metapath_analysis'  # Update to your Google Drive path
GENIZ_DIR  = os.path.join(BASE_DIR, 'GeniZ')
EMB_DIR    = os.path.join(BASE_DIR, 'metapath2vec_embeddings')

PAGERANK_PATH = os.path.join(GENIZ_DIR, 'pagerank_scores_zscore.pkl')
E2I_PATH      = os.path.join(GENIZ_DIR, 'entity2idx_zscore.pkl')

MODEL_DIR   = os.path.join(GENIZ_DIR, 'models_v2')
REPORTS_DIR = os.path.join(GENIZ_DIR, 'reports_v2')
os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Model
EMB_DIM    = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT    = 0.3
BATCH_SIZE = 512

# Training
LR           = 1.5e-4
WEIGHT_DECAY = 5e-5
EPOCHS_LITE  = 30
EPOCHS_FINAL = 40
FREEZE_CENT  = 8
WARMUP       = 5

# Band weights (PageRank z-score thresholds)
THR_LOW  = 0.114722   # p99
W_LOW    = 6.0
THR_HIGH = 0.610942   # p99.9
W_HIGH   = 16.0

# BPR inertia
BPR_MAX                 = 0.075
BPR_STEP                = 0.015
BPR_HOLD_EPOCHS         = 4
BPR_COOLDOWN_PATIENCE   = 4
BPR_ACTIVATION_PATIENCE = 5
ATTN_STABLE_TOL         = 0.015
ATTN_STABLE_WINDOW      = 3

# Splits & selection
TRAIN_FRAC     = 0.70
VAL_FRAC       = 0.15
DELTA_NDCG_MIN = 0.002

print('✅ Configuration set')

## 3. Metapath Schema Reference

In [ ]:
SCHEMA_MAP = {
    'metapath_1_emb' : 'Firm - FoS - Firm',
    'metapath_2_emb' : 'Firm - FoS - FoS - Firm',
    'metapath_3_emb' : 'FoS - Firm - FoS',
    'metapath_4_emb' : 'Firm - Patent - Country - Patent - Firm',
    'metapath_5_emb' : 'Firm - Product - FoS - Product - Firm',
    'metapath_6_emb' : 'Product - FoS - Product',
    'metapath_7_emb' : 'Firm - Country - Firm',
    'metapath_8_emb' : 'Firm - Univ - Country - Univ - Firm',
    'metapath_9_emb' : 'Patent - Country - Patent',
    'metapath_10_emb': 'FoS - Product - FoS',
    'metapath_11_emb': 'Product - FoS - Firm - FoS - Product',
    'metapath_12_emb': 'Product - Firm - Product',
    'metapath_13_emb': 'Univ - Firm - Univ',
    'metapath_14_emb': 'FoS - FoS - FoS',
}

def mp_label(tag):
    return tag.replace('_emb','').replace('metapath_','MP')

for tag, schema in SCHEMA_MAP.items():
    print(f'  {mp_label(tag):>5} -> {schema}')

## 4. Load Data

In [ ]:
with open(PAGERANK_PATH, 'rb') as f:
    pagerank_zscore = pickle.load(f)

zvals = np.array(list(pagerank_zscore.values()))
print(f'PageRank z-scores : {len(pagerank_zscore):,} nodes')
print(f'  min={zvals.min():.4f}  max={zvals.max():.2f}  '
      f'mean={zvals.mean():.6f}  std={zvals.std():.4f}')
print(f'  p99={np.percentile(zvals,99):.4f}  p99.9={np.percentile(zvals,99.9):.4f}')

In [ ]:
with open(E2I_PATH, 'rb') as f:
    entity2idx = pickle.load(f)
entity2idx_rev = {v: k for k, v in entity2idx.items()}
print(f'entity2idx : {len(entity2idx):,} nodes')

In [ ]:
def load_embeddings(emb_dir, allowed=None):
    embeddings, node_maps = {}, {}
    for fname in sorted(os.listdir(emb_dir)):
        if not fname.endswith('.pkl'):
            continue
        tag = fname.replace('.pkl', '')
        if allowed and tag not in allowed:
            continue
        with open(os.path.join(emb_dir, fname), 'rb') as f:
            data = pickle.load(f)
        emb = torch.tensor(data['embeddings'], dtype=torch.float32)
        id2node = data['id2node']
        node2row = {v: k for k, v in (
            id2node.items() if isinstance(id2node, dict) else enumerate(id2node))}
        embeddings[tag] = emb
        node_maps[tag]  = node2row
    print(f'Loaded {len(embeddings)} embeddings:')
    for tag, emb in embeddings.items():
        print(f'  {mp_label(tag):>5}: {emb.shape[0]:>7,} nodes x {emb.shape[1]} dims')
    return embeddings, node_maps

emb_all, map_all = load_embeddings(EMB_DIR)
MP_KEYS_ALL = sorted(emb_all.keys())

## 5. Dataset and DataLoader Utilities

In [ ]:
class GENIDataset(Dataset):
    """
    Node-level dataset for GeniZ.
    Missing nodes (not covered by a metapath) receive a zero embedding.
    """
    def __init__(self, pagerank, entity2idx, embeddings, node_maps, allowed=None):
        self.entity2idx = entity2idx
        self.embeddings = embeddings
        self.node_maps  = node_maps
        self.mp_keys    = allowed or sorted(embeddings.keys())
        self.emb_dim    = next(iter(embeddings.values())).shape[1]
        self.node_ids   = [nid for nid in entity2idx if nid in pagerank]
        self.targets    = {nid: pagerank[nid] for nid in self.node_ids}
        print(f'GENIDataset: {len(self.node_ids):,} nodes | {len(self.mp_keys)} metapaths')

    def __len__(self): return len(self.node_ids)

    def __getitem__(self, idx):
        nid  = self.node_ids[idx]
        nidx = self.entity2idx[nid]
        y    = self.targets[nid]
        emb_dict = {}
        for tag in self.mp_keys:
            row = self.node_maps[tag].get(nid)
            emb_dict[tag] = (self.embeddings[tag][row] if row is not None
                             else torch.zeros(self.emb_dim))
        return emb_dict, nidx, float(y), 1.0


def collate_geni(batch):
    mp_keys  = list(batch[0][0].keys())
    emb_b    = {k: torch.stack([b[0][k] for b in batch]) for k in mp_keys}
    node_ids = torch.tensor([b[1] for b in batch], dtype=torch.long)
    y_b      = torch.tensor([b[2] for b in batch], dtype=torch.float32)
    w_b      = torch.tensor([b[3] for b in batch], dtype=torch.float32)
    return emb_b, node_ids, y_b, w_b


def make_splits(dataset, train_frac=0.70, val_frac=0.15, seed=42):
    N   = len(dataset)
    idx = np.random.default_rng(seed).permutation(N)
    n_tr, n_va = int(N*train_frac), int(N*val_frac)
    tr = Subset(dataset, idx[:n_tr])
    va = Subset(dataset, idx[n_tr:n_tr+n_va])
    te = Subset(dataset, idx[n_tr+n_va:])
    print(f'Split: train={len(tr):,} | val={len(va):,} | test={len(te):,}')
    return tr, va, te


def band_sampler(dataset, subset):
    zs = np.array([dataset.targets[dataset.node_ids[i]] for i in subset.indices])
    w  = np.ones_like(zs, dtype=np.float32)
    w[zs > THR_HIGH] = W_HIGH
    w[(zs > THR_LOW) & (zs <= THR_HIGH)] = W_LOW
    return WeightedRandomSampler(torch.tensor(w), len(w))


def make_loaders(dataset, train_set, val_set, test_set):
    tr = DataLoader(train_set, BATCH_SIZE, sampler=band_sampler(dataset, train_set),
                    collate_fn=collate_geni)
    va = DataLoader(val_set,   BATCH_SIZE, shuffle=False, collate_fn=collate_geni)
    te = DataLoader(test_set,  BATCH_SIZE, shuffle=False, collate_fn=collate_geni)
    return tr, va, te


print('Dataset utilities defined')

# Build dataset with all 14 metapaths
dataset_all = GENIDataset(pagerank_zscore, entity2idx, emb_all, map_all, MP_KEYS_ALL)
train_all, val_all, test_all = make_splits(dataset_all, TRAIN_FRAC, VAL_FRAC, SEED)
loader_tr_all, loader_va_all, loader_te_all = make_loaders(
    dataset_all, train_all, val_all, test_all)
print('Loaders ready (14 metapaths)')

## 6. GeniModel Architecture

In [ ]:
class GeniModel(nn.Module):
    """
    GeniZ heterogeneous attention network for characterising contextual centrality.
    Adaptation of GENI (Park et al., 2019) for heterogeneous graphs.

    Architecture:
    - Per-metapath MLP: L2-normalised embedding -> hidden (x num_layers) -> scalar
    - Masked softmax attention (zero-coverage nodes excluded per metapath)
    - Per-node centrality bias (frozen for first FREEZE_CENT epochs)
    """
    def __init__(self, metapath_names, num_nodes,
                 embedding_dim=128, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.metapath_names = metapath_names
        self.num_nodes      = num_nodes

        self.metapath_mlps = nn.ModuleDict({
            name: nn.ModuleList(
                [nn.Sequential(
                    nn.Linear(embedding_dim if i == 0 else hidden_dim, hidden_dim),
                    nn.ReLU(), nn.Dropout(dropout)
                ) for i in range(num_layers)]
                + [nn.Linear(hidden_dim, 1)]
            )
            for name in metapath_names
        })

        self.attn_logits       = nn.Parameter(torch.randn(len(metapath_names)))
        self.centrality_params = nn.Parameter(torch.zeros(num_nodes, 1))
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)

    def get_attention_weights(self):
        with torch.no_grad():
            w = torch.softmax(self.attn_logits.cpu(), dim=0).numpy()
        return {name: float(w[i]) for i, name in enumerate(self.metapath_names)}

    def forward(self, emb_dict, node_ids):
        scores, masks = [], []
        for name in self.metapath_names:
            x     = F.normalize(emb_dict[name], p=2, dim=1)
            valid = (x.abs().sum(dim=1) > 1e-6).float()
            for layer in self.metapath_mlps[name][:-1]:
                x = layer(x)
            scores.append(self.metapath_mlps[name][-1](x).squeeze(-1))
            masks.append(valid)

        S = torch.stack(scores, dim=1)
        M = torch.stack(masks,  dim=1)
        logits = self.attn_logits.unsqueeze(0).expand_as(S)
        logits = logits * M + (1 - M) * (-1e9)
        A      = F.softmax(logits, dim=1)
        y_hat  = (A * S).sum(dim=1)
        y_hat  = y_hat + self.centrality_params[node_ids.long()].squeeze(1)
        return y_hat


print('GeniModel defined')

## 7. Training and Evaluation Utilities

In [ ]:
def band_weights_tensor(z):
    w = torch.ones_like(z)
    w = torch.where(z > THR_HIGH, torch.as_tensor(W_HIGH, device=z.device, dtype=z.dtype), w)
    w = torch.where((z > THR_LOW) & (z <= THR_HIGH),
                    torch.as_tensor(W_LOW, device=z.device, dtype=z.dtype), w)
    return w


def train_epoch(model, loader, optimiser, loss_fn, bpr_on=False, bpr_coeff=0.0):
    """
    One training epoch.
    Loss = HuberLoss (band-weighted) + L1 attn + L2 centrality + BPR (when active).
    """
    model.train()
    total = 0.0
    for emb_dict, node_ids, y, _ in tqdm(loader, leave=False):
        node_ids = node_ids.to(device).long()
        y = y.to(device)
        for k in emb_dict: emb_dict[k] = emb_dict[k].to(device)

        y_hat = model(emb_dict, node_ids)
        raw   = loss_fn(y_hat, y)
        l1    = 5e-6 * model.attn_logits.abs().sum()
        l2    = 1e-5 * model.centrality_params.pow(2).sum()
        loss  = (raw * band_weights_tensor(y)).mean() + l1 + l2

        if bpr_on and bpr_coeff > 0:
            pos = y > THR_HIGH
            neg = y <= THR_LOW
            if pos.any() and neg.any():
                yp, yn  = y_hat[pos], y_hat[neg]
                k_p = min(512, yp.shape[0] * yn.shape[0])
                ip  = torch.randint(0, yp.shape[0], (k_p,), device=device)
                ig  = torch.randint(0, yn.shape[0], (k_p,), device=device)
                bpr = -torch.log(torch.sigmoid(yp[ip] - yn[ig]) + 1e-8).mean()
                loss = loss + bpr_coeff * bpr

        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimiser.step()
        total += loss.item()
    return total / max(1, len(loader))


def rbp_score(y_true, y_pred, p=0.95):
    """Rank-Biased Precision (Moffat & Zobel, 2008)."""
    order    = np.argsort(y_pred)[::-1]
    y_sorted = y_true[order]
    mn, mx   = y_sorted.min(), y_sorted.max()
    rel      = (y_sorted - mn) / (mx - mn) if mx > mn else np.ones_like(y_sorted)
    ranks    = np.arange(1, len(rel) + 1)
    return float((1 - p) * np.sum(rel * (p ** (ranks - 1))))


def evaluate(model, loader, perturb_mp=None):
    """
    Compute metrics.
    Primary: Spearman, NDCG@100, combined = (Spearman + NDCG) / 2
    Diagnostic: R2, RBP(0.95), RBP(0.80)
    """
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for emb_dict, node_ids, y, _ in loader:
            if perturb_mp and perturb_mp in emb_dict:
                emb_dict[perturb_mp] = emb_dict[perturb_mp][
                    torch.randperm(emb_dict[perturb_mp].size(0))]
            node_ids = node_ids.to(device).long()
            for k in emb_dict: emb_dict[k] = emb_dict[k].to(device)
            all_pred.extend(model(emb_dict, node_ids).cpu().numpy())
            all_true.extend(y.numpy())

    yt  = np.array(all_true)
    yp  = np.array(all_pred)
    ytn = yt - yt.min()
    spr = float(spearmanr(yt, yp).statistic)
    nd  = float(ndcg_score(ytn.reshape(1,-1), yp.reshape(1,-1), k=100))
    return {
        'spearman': spr, 'ndcg': nd, 'combined': (spr+nd)/2.0,
        'r2': float(r2_score(yt, yp)),
        'rbp_95': rbp_score(yt, yp, 0.95),
        'rbp_80': rbp_score(yt, yp, 0.80),
    }


print('Training and evaluation utilities defined')

In [ ]:
def run_training(model, tr_loader, va_loader, epochs, name):
    """
    Full training loop:
    - Linear LR warm-up for WARMUP epochs
    - Centrality frozen for FREEZE_CENT epochs
    - BPR with inertia (activates when NDCG plateaus + attention stable)
    - Saves best NDCG and best Combined checkpoints
    """
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    lf    = nn.HuberLoss(reduction='none', delta=1.0)

    ck_nd   = os.path.join(MODEL_DIR, f'{name}_best_ndcg.pt')
    ck_comb = os.path.join(MODEL_DIR, f'{name}_best_combined.pt')

    best_nd = -1.0; best_comb = -1.0; no_imp = 0
    bpr_on = False; bpr_c = 0.0; bpr_hold = 0
    attn_prev = None; attn_hist = []

    def get_attn():
        with torch.no_grad():
            return torch.softmax(model.attn_logits.cpu(), dim=0).numpy()

    def is_stable(d):
        attn_hist.append(d)
        if len(attn_hist) < ATTN_STABLE_WINDOW: return False
        return all(x <= ATTN_STABLE_TOL for x in attn_hist[-ATTN_STABLE_WINDOW:])

    def set_lr(lr):
        for pg in opt.param_groups: pg['lr'] = lr

    for epoch in range(1, epochs + 1):
        set_lr(LR * epoch / WARMUP if epoch <= WARMUP else LR)
        model.centrality_params.requires_grad_(epoch > FREEZE_CENT)

        tr_loss = train_epoch(model, tr_loader, opt, lf, bpr_on, bpr_c)
        m = evaluate(model, va_loader)

        av    = get_attn()
        delta = float(np.abs(av - attn_prev).sum()) if attn_prev is not None else float('nan')
        attn_prev = av.copy()
        ds = f'{delta:.4f}' if np.isfinite(delta) else 'nan'

        print(f'Ep {epoch:02d} | loss={tr_loss:.4f} | '
              f'spr={m["spearman"]:.4f} | ndcg={m["ndcg"]:.4f} | '
              f'comb={m["combined"]:.4f} | r2={m["r2"]:.4f} | '
              f'rbp95={m["rbp_95"]:.4f} | '
              f'BPR={"ON" if bpr_on else "OFF"}({bpr_c:.3f}) | da={ds}')

        sched.step(tr_loss)

        if m['ndcg'] > best_nd + 1e-6:
            best_nd = m['ndcg']; no_imp = 0
            torch.save(model.state_dict(), ck_nd)
            print(f'  Best NDCG  -> {best_nd:.4f}')
        else:
            no_imp += 1

        if m['combined'] > best_comb:
            best_comb = m['combined']
            torch.save(model.state_dict(), ck_comb)
            print(f'  Best Comb  -> {best_comb:.4f}')

        stable = is_stable(delta if np.isfinite(delta) else 1e9)
        if not bpr_on and no_imp >= BPR_ACTIVATION_PATIENCE and stable:
            bpr_on = True; bpr_c = BPR_STEP; bpr_hold = 1
            print(f'  [BPR] Activated coeff={bpr_c:.3f}')
        elif bpr_on:
            if bpr_hold < BPR_HOLD_EPOCHS:
                bpr_hold += 1
            elif m['ndcg'] > best_nd - 1e-6 and bpr_c < BPR_MAX:
                bpr_c = min(bpr_c + BPR_STEP, BPR_MAX)
                print(f'  [BPR] Ramp up coeff={bpr_c:.3f}')
            elif no_imp >= BPR_COOLDOWN_PATIENCE:
                bpr_c = max(0.0, bpr_c - BPR_STEP)
                if bpr_c == 0.0: bpr_on = False; print('  [BPR] Deactivated')

    print(f'Done. Best NDCG={best_nd:.4f} | Best Combined={best_comb:.4f}')
    return ck_nd, ck_comb


print('run_training defined')

## 8. GeniZ Lite — Exploratory (14 Metapaths)

Train a lightweight model on all 14 metapaths. The attention weights serve as an **exploratory signal** to inform the conceptual branch separation in §9.

> Note: The attention weights serve as an **exploratory signal** for which pathways carry non-redundant structural information about knowledge diffusion routes.

In [ ]:
for f in glob.glob(os.path.join(MODEL_DIR, 'GeniZ_Lite_*.pt')):
    os.remove(f); print(f'Removed: {f}')

model_lite = GeniModel(
    metapath_names=MP_KEYS_ALL, num_nodes=len(entity2idx),
    embedding_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)

print(f'Parameters: {sum(p.numel() for p in model_lite.parameters()):,}')
ck_lite_nd, ck_lite_comb = run_training(
    model_lite, loader_tr_all, loader_va_all, EPOCHS_LITE, 'GeniZ_Lite')

In [ ]:
model_lite.load_state_dict(torch.load(ck_lite_comb, map_location=device))
attn_lite = model_lite.get_attention_weights()

df_attn_lite = pd.DataFrame([
    {'Metapath': mp_label(tag), 'Schema': SCHEMA_MAP.get(tag,''),
     'Attention weight': round(w, 4)}
    for tag, w in sorted(attn_lite.items(), key=lambda x: -x[1])
])
print('=== GeniZ Lite Attention Weights (exploratory signal) ===')
print(df_attn_lite.to_string(index=False))
df_attn_lite.to_csv(os.path.join(REPORTS_DIR, 'attn_weights_lite.csv'), index=False)
print('Saved: attn_weights_lite.csv')

## 9. Knowledge Pathway Branches: Cognitive vs. Institutional Proximity

Following Boschma (2005), knowledge diffusion mechanisms operate through
two distinct proximity dimensions

- **Rama A — Cognitive proximity**: pathways through Fields of Science (FoS)
  and Products — encode similarity in firms' knowledge base and technological capabilities
- **Rama B — Institutional proximity**: pathways through Patents, Countries,
  and Universities — encode embeddedness in shared innovation ecosystems

GeniZ Lite attention weights corroborate this separation.

In [ ]:
RAMA_A = [
    'metapath_1_emb',    # Firm - FoS - Firm
    'metapath_2_emb',    # Firm - FoS - FoS - Firm
    'metapath_3_emb',    # FoS - Firm - FoS
    'metapath_5_emb',    # Firm - Product - FoS - Product - Firm
    'metapath_6_emb',    # Product - FoS - Product
    'metapath_10_emb',   # FoS - Product - FoS
    'metapath_11_emb',   # Product - FoS - Firm - FoS - Product
    'metapath_14_emb',   # FoS - FoS - FoS
]
RAMA_B = [
    'metapath_4_emb',    # Firm - Patent - Country - Patent - Firm
    'metapath_7_emb',    # Firm - Country - Firm
    'metapath_8_emb',    # Firm - Univ - Country - Univ - Firm
    'metapath_9_emb',    # Patent - Country - Patent
    'metapath_12_emb',   # Product - Firm - Product
    'metapath_13_emb',   # Univ - Firm - Univ
]

print('=== Rama A - Knowledge Proximity ===')
for mp in RAMA_A:
    print(f'  {mp_label(mp):>5} | attn_lite={attn_lite.get(mp,0):.4f} | {SCHEMA_MAP[mp]}')

print('\n=== Rama B - Institutional Proximity ===')
for mp in RAMA_B:
    print(f'  {mp_label(mp):>5} | attn_lite={attn_lite.get(mp,0):.4f} | {SCHEMA_MAP[mp]}')

## 10. Forward Selection Within Each Branch

Greedy forward selection within each branch. A **Ridge regression** is used as a fast proxy for GeniZ to avoid retraining the full neural model at each step. Metapaths are added while ΔNDCG > 0.002.

In [ ]:
def get_features(mp_list):
    common = sorted(
        set.intersection(*[set(map_all[t].keys()) for t in mp_list])
        & set(pagerank_zscore.keys()))
    if len(common) < 500: return None, None
    Xs = []
    for t in mp_list:
        e = emb_all[t].numpy()[[map_all[t][n] for n in common]]
        norms = np.linalg.norm(e, axis=1, keepdims=True).clip(1e-8)
        Xs.append(e / norms)
    X = np.concatenate(Xs, axis=1)
    y = np.array([pagerank_zscore[n] for n in common])
    return X, y


def score_ridge(mp_list):
    X, y = get_features(mp_list)
    if X is None: return -1.0, -1.0
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=SEED)
    pipe = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
    pipe.fit(X_tr, y_tr)
    yp  = pipe.predict(X_va)
    nd  = float(ndcg_score([y_va - y_va.min()], [yp], k=100))
    spr = float(spearmanr(y_va, yp).statistic)
    return nd, spr


def forward_select(candidates, branch_name):
    print(f'\n=== Forward Selection - {branch_name} ===')
    selected = []; remaining = candidates.copy(); baseline = 0.0; log = []

    while remaining:
        best_mp = best_nd = best_spr = best_gain = None
        best_nd = baseline; best_gain = -1.0

        for mp in remaining:
            nd, spr = score_ridge(selected + [mp])
            gain = nd - baseline
            if gain > best_gain:
                best_mp, best_nd, best_spr, best_gain = mp, nd, spr, gain

        if best_mp is None or best_gain < DELTA_NDCG_MIN:
            print('No more candidates above threshold.')
            break

        selected.append(best_mp); remaining.remove(best_mp); baseline = best_nd
        print(f'  + {mp_label(best_mp):>5} ({SCHEMA_MAP.get(best_mp,"")}): '
              f'NDCG={best_nd:.4f} (D={best_gain:+.4f}) | Spr={best_spr:.4f}')
        log.append({'branch': branch_name, 'metapath': mp_label(best_mp),
                    'schema': SCHEMA_MAP.get(best_mp,''),
                    'ndcg': round(best_nd,4), 'delta_ndcg': round(best_gain,4),
                    'spearman': round(best_spr,4)})

    print(f'Selected: {[mp_label(s) for s in selected]}')
    return selected, log


selected_A, log_A = forward_select(RAMA_A, 'Rama A - Knowledge')
selected_B, log_B = forward_select(RAMA_B, 'Rama B - Institutional')

df_fs = pd.DataFrame(log_A + log_B)
df_fs.to_csv(os.path.join(REPORTS_DIR, 'forward_selection.csv'), index=False)
print('\nSaved: forward_selection.csv')

In [ ]:
# Rama A validated by forward selection (Ridge proxy works well for dense FoS/Product)
# Rama B selected by GeniZ Lite attention threshold (Ridge underestimates sparse patent/univ metapaths)

ATTN_THRESHOLD_B = 0.02

selected_B_lite = [
    mp for mp in RAMA_B
    if attn_lite.get(mp, 0) >= ATTN_THRESHOLD_B
]

print('=== Rama B selection by Lite attention (threshold > 0.02) ===')
for mp in RAMA_B:
    aw = attn_lite.get(mp, 0)
    status = 'INCLUDED' if aw >= ATTN_THRESHOLD_B else 'excluded'
    print(f'  {mp_label(mp):>5}: attn={aw:.4f} -> {status}')

print(f'\nForward selection gave   : {[mp_label(m) for m in selected_B]}')
print(f'Lite attention gives     : {[mp_label(m) for m in selected_B_lite]}')
print(f'\nUsing Lite attention selection for Rama B.')

# Override
selected_B = selected_B_lite

In [ ]:
MP_KEYS_FINAL = selected_A + selected_B

print('=== Final Metapath Set ===')
print(f'Rama A ({len(selected_A)}): {[mp_label(m) for m in selected_A]}')
print(f'Rama B ({len(selected_B)}): {[mp_label(m) for m in selected_B]}')
print(f'Total : {len(MP_KEYS_FINAL)} metapaths\n')
for mp in MP_KEYS_FINAL:
    branch = 'A' if mp in RAMA_A else 'B'
    print(f'  [{branch}] {mp_label(mp):>5} -> {SCHEMA_MAP[mp]}')

## 11. GeniZ Final — Selected Metapaths

The final model is trained on the metapaths selected by forward selection across both branches. Evaluation metrics on the held-out test set are reported in the paper.

In [ ]:
emb_fin = {k: emb_all[k] for k in MP_KEYS_FINAL}
map_fin = {k: map_all[k] for k in MP_KEYS_FINAL}

dataset_fin = GENIDataset(pagerank_zscore, entity2idx, emb_fin, map_fin, MP_KEYS_FINAL)
train_fin, val_fin, test_fin = make_splits(dataset_fin, TRAIN_FRAC, VAL_FRAC, SEED)
loader_tr_fin, loader_va_fin, loader_te_fin = make_loaders(
    dataset_fin, train_fin, val_fin, test_fin)
print('Final DataLoaders ready')

In [ ]:
for f in glob.glob(os.path.join(MODEL_DIR, 'GeniZ_Final_*.pt')):
    os.remove(f); print(f'Removed: {f}')

model_final = GeniModel(
    metapath_names=MP_KEYS_FINAL, num_nodes=len(entity2idx),
    embedding_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)

print(f'GeniZ Final parameters: {sum(p.numel() for p in model_final.parameters()):,}')
ck_fin_nd, ck_fin_comb = run_training(
    model_final, loader_tr_fin, loader_va_fin, EPOCHS_FINAL, 'GeniZ_Final')

## 12. Test Set Evaluation

**Metrics**:
- **Spearman rho**: rank correlation across the full distribution (primary)
- **NDCG@100**: identifies the top-100 innovators from 1.2M nodes (primary)
- **Combined**: (Spearman + NDCG) / 2 — training objective
- **RBP(p=0.95)**: full-ranking emphasis (diagnostic)
- **RBP(p=0.80)**: top-k emphasis (diagnostic)
- **R²**: diagnostic only — low values expected given the power-law distribution of contextual centrality; findings are interpreted as configurational, not predictive

In [ ]:
model_final.load_state_dict(torch.load(ck_fin_comb, map_location=device))
test_m = evaluate(model_final, loader_te_fin)

print('=== Test Set Evaluation - GeniZ Final ===')
print(f'  Spearman rho : {test_m["spearman"]:.4f}')
print(f'  NDCG@100     : {test_m["ndcg"]:.4f}')
print(f'  Combined     : {test_m["combined"]:.4f}')
print(f'  RBP (p=0.95) : {test_m["rbp_95"]:.4f}')
print(f'  RBP (p=0.80) : {test_m["rbp_80"]:.4f}')
print(f'  R2           : {test_m["r2"]:.4f}')

pd.DataFrame([test_m]).to_csv(os.path.join(REPORTS_DIR, 'test_metrics.csv'), index=False)
print('\nSaved: test_metrics.csv')

In [ ]:
attn_final = model_final.get_attention_weights()

df_attn_final = pd.DataFrame([
    {'Metapath': mp_label(tag), 'Schema': SCHEMA_MAP.get(tag,''),
     'Branch': 'A' if tag in RAMA_A else 'B',
     'Attention weight': round(w, 4)}
    for tag, w in sorted(attn_final.items(), key=lambda x: -x[1])
])
print('=== GeniZ Final - Attention Weights ===')
print(df_attn_final.to_string(index=False))
df_attn_final.to_csv(os.path.join(REPORTS_DIR, 'attn_weights_final.csv'), index=False)
print('Saved: attn_weights_final.csv')

## 13. Permutation Feature Importance

Contribution of each metapath measured by shuffling its embeddings at inference. Drop in NDCG and Spearman indicates non-redundant information carried by that metapath. -> Table 4 of the paper.

In [ ]:
baseline = evaluate(model_final, loader_te_fin)
print(f'Baseline: spr={baseline["spearman"]:.4f} | '
      f'ndcg={baseline["ndcg"]:.4f} | rbp95={baseline["rbp_95"]:.4f}\n')

rows = []
for mp in tqdm(MP_KEYS_FINAL, desc='Permuting'):
    m = evaluate(model_final, loader_te_fin, perturb_mp=mp)
    drop = lambda a, b: (a - b) / abs(a) * 100 if abs(a) > 1e-9 else 0.0
    rows.append({
        'Metapath'         : mp_label(mp),
        'Schema'           : SCHEMA_MAP[mp],
        'Branch'           : 'A' if mp in RAMA_A else 'B',
        'NDCG drop (%)'    : round(drop(baseline['ndcg'],     m['ndcg']),     2),
        'Spearman drop (%)': round(drop(baseline['spearman'], m['spearman']), 2),
        'RBP drop (%)'     : round(drop(baseline['rbp_95'],   m['rbp_95']),   2),
    })
    print(f'  {mp_label(mp):>5}: '
          f'NDCG drop={rows[-1]["NDCG drop (%)"]:.2f}% | '
          f'Spr drop={rows[-1]["Spearman drop (%)"]:.2f}%')

df_pfi = pd.DataFrame(rows).sort_values('NDCG drop (%)', ascending=False)
print('\n=== Ranked by NDCG drop ===')
print(df_pfi.to_string(index=False))
df_pfi.to_csv(os.path.join(REPORTS_DIR, 'permutation_importance.csv'), index=False)
print('\nSaved: permutation_importance.csv')

## 14. System Contour: Core-Periphery via GMM

Identifies the structural contour of the European innovation system through
**contextual centrality zones**. Consistent with Borgatti & Everett (2000),
coreness is treated as a continuous property — not a binary classification.

In [ ]:
# Reload after disconnection
model_final.load_state_dict(torch.load(ck_fin_comb, map_location=device))
model_final.eval()
print('Model reloaded')

In [ ]:
model_final.eval()
all_nidx, all_preds, all_true = [], [], []
full_loader = DataLoader(dataset_fin, BATCH_SIZE, shuffle=False, collate_fn=collate_geni)

with torch.no_grad():
    for emb_dict, node_ids, y, _ in tqdm(full_loader, desc='Predicting'):
        node_ids = node_ids.to(device).long()
        for k in emb_dict: emb_dict[k] = emb_dict[k].to(device)
        all_preds.extend(model_final(emb_dict, node_ids).cpu().numpy())
        all_nidx.extend(node_ids.cpu().numpy())
        all_true.extend(y.numpy())

df_rank = pd.DataFrame({
    'node_idx'     : all_nidx,
    'global_id'    : [entity2idx_rev[i] for i in all_nidx],
    'pagerank_true': all_true,
    'score_pred'   : all_preds,
})
df_rank['prefix']    = df_rank['global_id'].str.extract(r'^([a-zA-Z]+)')  # fix: closing )
df_rank['rank_true'] = df_rank['pagerank_true'].rank(ascending=False).astype(int)
df_rank['rank_pred'] = df_rank['score_pred'].rank(ascending=False).astype(int)

print(f'Predictions: {len(df_rank):,} nodes')
print(df_rank['prefix'].value_counts())

In [ ]:
df_firms = df_rank[df_rank['prefix'] == 'firm'].copy()
scores   = df_firms['score_pred'].values.reshape(-1, 1)

bic = {}
for n in [2, 3, 4]:
    g = GaussianMixture(n_components=n, random_state=SEED, n_init=5)
    g.fit(scores)
    bic[n] = g.bic(scores)
    print(f'  GMM n={n}: BIC={bic[n]:.1f}')

best_n = min(bic, key=bic.get)
print(f'Best n_components = {best_n}')

gmm    = GaussianMixture(n_components=best_n, random_state=SEED, n_init=10)
gmm.fit(scores)
labels = gmm.predict(scores)
means  = gmm.means_.flatten()
order  = np.argsort(means)[::-1]

# Fix: enough zone names for any n up to 4
zone_names_all = ['core', 'semi-core', 'semi-periphery', 'periphery']
zones = zone_names_all[:best_n]
c2z   = {comp: zones[rank] for rank, comp in enumerate(order)}

df_firms = df_firms.copy()
df_firms['zone'] = [c2z[c] for c in labels]

print('\n=== Core-Periphery Distribution (firms) ===')
for zone in zones:
    n   = (df_firms['zone'] == zone).sum()
    pct = n / len(df_firms) * 100
    mpr = df_firms[df_firms['zone'] == zone]['pagerank_true'].mean()
    print(f'  {zone:<14}: {n:>7,}  ({pct:5.1f}%)  mean PR z-score={mpr:.4f}')

df_firms.to_csv(os.path.join(REPORTS_DIR, 'core_periphery_gmm.csv'), index=False)
print('\nSaved: core_periphery_gmm.csv')

In [ ]:
colours = {'core': '#e74c3c', 'semi-core': '#e67e22',
           'semi-periphery': '#f1c40f', 'periphery': '#3498db'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1 — full distribution (log scale y)
for zone in zones:
    sub = df_firms[df_firms['zone'] == zone]['score_pred']
    axes[0].hist(sub, bins=80, alpha=0.6, label=zone, color=colours[zone])
axes[0].set_yscale('log')
axes[0].set_xlabel('GeniZ predicted score')
axes[0].set_ylabel('Firm count (log)')
axes[0].set_title('Full distribution (log scale)')
axes[0].legend(fontsize=8)

# Panel 2 — zoomed on non-core (score < 5) to see the bulk
for zone in [z for z in zones if z != 'core']:
    sub = df_firms[(df_firms['zone'] == zone) & (df_firms['score_pred'] < 5)]['score_pred']
    axes[1].hist(sub, bins=60, alpha=0.6, label=zone, color=colours[zone])
axes[1].set_xlabel('GeniZ predicted score')
axes[1].set_ylabel('Firm count')
axes[1].set_title('Non-core firms (score < 5)')
axes[1].legend(fontsize=8)

# Panel 3 — zone sizes
sizes = [int((df_firms['zone'] == z).sum()) for z in zones]
bars = axes[2].bar(zones, sizes, color=[colours[z] for z in zones])
axes[2].set_ylabel('Number of firms')
axes[2].set_title('Zone sizes')
for i, v in enumerate(sizes):
    axes[2].text(i, v + 200, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'core_periphery_gmm.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: core_periphery_gmm.png')

In [ ]:
import shutil
from datetime import datetime

# Timestamp para identificar esta corrida
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M')
print(f'Run ID: {RUN_ID}')

# ── Verificar que todos los archivos clave existen ─────────────────────────
files_expected = [
    'attn_weights_lite.csv',
    'forward_selection.csv',
    'attn_weights_final.csv',
    'test_metrics.csv',
    'permutation_importance.csv',
    'core_periphery_gmm.csv',
    'core_periphery_gmm.png',
]

print('\n=== Reports ===')
for fname in files_expected:
    path = os.path.join(REPORTS_DIR, fname)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f'  {"OK" if exists else "MISSING":6} {fname} ({size:,} bytes)')

print('\n=== Model checkpoints ===')
for fname in os.listdir(MODEL_DIR):
    if fname.endswith('.pt'):
        path = os.path.join(MODEL_DIR, fname)
        size = os.path.getsize(path)
        print(f'  OK     {fname} ({size/1e6:.1f} MB)')

In [ ]:
import numpy as np

def to_python(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f'Not serializable: {type(obj)}')

# Summary JSON — single source of truth for the paper
summary = {
    'run_id'          : RUN_ID,
    'metapaths_lite'  : MP_KEYS_ALL,
    'rama_a'          : selected_A,
    'rama_b'          : selected_B,
    'mp_keys_final'   : MP_KEYS_FINAL,
    'attn_lite'       : {mp_label(k): round(v,4) for k,v in attn_lite.items()},
    'attn_final'      : {mp_label(k): round(v,4) for k,v in attn_final.items()},
    'test_metrics'    : test_m,
    'core_periphery'  : {
        zone: {
            'n'             : int((df_firms['zone']==zone).sum()),
            'pct'           : round(float((df_firms['zone']==zone).sum()/len(df_firms)*100), 2),
            'mean_pr_zscore': round(float(df_firms[df_firms['zone']==zone]['pagerank_true'].mean()), 4)
        }
        for zone in zones
    },
    'gmm_n_components': int(best_n),
    'pfi'             : df_pfi[['Metapath','NDCG drop (%)','Spearman drop (%)']].to_dict('records'),
}

summary_path = os.path.join(REPORTS_DIR, f'run_summary_{RUN_ID}.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=to_python)

print(f'Summary saved: run_summary_{RUN_ID}.json')
print(f'\nAll results in: {REPORTS_DIR}')
print(f'All models  in: {MODEL_DIR}')

print('\n' + '='*50)
print('KEY NUMBERS FOR THE PAPER')
print('='*50)
print(f'  Spearman rho   : {test_m["spearman"]:.4f}')
print(f'  NDCG@100       : {test_m["ndcg"]:.4f}')
print(f'  RBP (p=0.95)   : {test_m["rbp_95"]:.4f}')
print(f'  RBP (p=0.80)   : {test_m["rbp_80"]:.4f}')
print(f'  Metapaths final: {len(MP_KEYS_FINAL)} ({[mp_label(m) for m in MP_KEYS_FINAL]})')
print(f'  Core firms     : {int((df_firms["zone"]=="core").sum()):,} ({round((df_firms["zone"]=="core").sum()/len(df_firms)*100,1)}%)')
print(f'  GMM components : {best_n}')
print('='*50)